# Performance Analysis Report

This notebook provides comprehensive performance analysis for the Midnight Performance network, including:
- Transaction throughput and validation metrics
- Mempool behavior and resource consumption
- Block production, import and size analysis
- Chain fork detection

---

## 1. Setup & Configuration

In [ ]:
import os
import importlib
import sys
import subprocess
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from datetime import datetime

# Add benchmark utils to path
for path in ["../block_size_benchmarks/block_size_analysis", "../traffic_benchmarks", "../fork_benchmarks", ".."]:
    abs_path = os.path.abspath(path)
    if abs_path not in sys.path:
        sys.path.append(abs_path)

import fetch_block_sizes
import extract_block_range_from_logs
from traffic_benchmarks import traffic_analyzer
from mempool_benchmarks import extractor, analyzer
from fork_benchmarks import fork_analyzer

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

# --- CONFIGURATION ---
# Define how many log files you expect (e.g., number of nodes in your cluster)
EXPECTED_NODE_COUNT = 20 

# Define time range for log download
feb4_3hours = {"from":"2026-02-04 18:14:17","to":"2026-02-04 21:10:33"} 
feb5_fund96_oscar_fixes = {"from":"2026-02-05 11:29:53","to":"2026-02-05 11:39:23"}
feb5_load28minutes_10txsper6secv = {"from":"2026-02-05 20:16:12","to":"2026-02-05 20:44:20"}
feb5_load1hour_3txsper6sec = {"from":"2026-02-05 21:15:15","to":"2026-02-05 22:24:20"}

TIME_RANGE = feb5_load1hour_3txsper6sec
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'logs'))

# Parse start and end times
start_time = TIME_RANGE['from']
end_time = TIME_RANGE['to']

# Assuming script is in parent dir relative to this notebook
script_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
script_path = os.path.join(script_dir, 'download_logs.py')

# Construct LOG_DIR based on time range (matching download_logs.py convention)
def sanitize_time(t):
    return t.replace(':', '-').replace(' ', '_')

folder_name = f"from_{sanitize_time(start_time)}_to_{sanitize_time(end_time)}"
LOG_DIR = os.path.join(BASE_PATH, folder_name)
print(f"\nLOG_DIR set to: {LOG_DIR}")
print(f"Targeting logs for range: {start_time} to {end_time}")

# --- UPDATED CHECK LOGIC ---
should_download = True

if os.path.exists(LOG_DIR):
    # Check specifically for .txt files
    files = [f for f in os.listdir(LOG_DIR) if f.endswith('.txt')]
    file_count = len(files)
    
    # If we have the exact number (or more), we skip.
    # If we have less (e.g. 18 out of 20), we force the download to run again.
    if file_count >= EXPECTED_NODE_COUNT:
        print(f"\nLogs already exist and appear complete ({file_count}/{EXPECTED_NODE_COUNT}).")
        print("Skipping download.")
        should_download = False
    else:
        print(f"\nDirectory exists but is incomplete ({file_count}/{EXPECTED_NODE_COUNT} files found).")
        print("Running download script to fetch missing logs...")
        should_download = True
else:
    print(f"\nDirectory not found. Downloading logs...")
    should_download = True

# Passing BASE_PATH as output-dir because the script appends the date folder
cmd = [
    sys.executable, script_path,
    "--from-time", start_time,
    "--to-time", end_time,
    "--output-dir", BASE_PATH
]

# We run with check=False so we can print output even on failure
# IMPORTANT: Run from the script directory so relative paths (like config) work
if should_download:
    try:
        process = subprocess.Popen(cmd, cwd=script_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='')
        process.wait()
        if process.returncode != 0:
            raise RuntimeError(f"Download script failed with code {process.returncode}")
        
        # Verify result after run
        final_files = [f for f in os.listdir(LOG_DIR) if f.endswith('.txt')]
        print(f"\nDownload finished. Total files: {len(final_files)}/{EXPECTED_NODE_COUNT}")
        
    except Exception as e:
        print(f"An error occurred during execution: {e}")

In [ ]:
# --- NODE DEFINITIONS ---
BLOCK_PRODUCERS = ["alice", "bob", "charlie", "dave", "eve", "kate", "leo", "mike", "nina", "oliver"]
RELAYS = ["ferdie", "george", "henry", "iris", "jack", "paul", "quinn", "rita", "sam", "tom"]
ALL_NODES = BLOCK_PRODUCERS + RELAYS
TX_POOL_NODES = ['charlie', 'ferdie'] # Validator + Relay

#===================================================================================\n# Count number of distinct txs in logs\n#===================================================================================\\

# ---------------------------------------------------------
# STEP 2: RUN ANALYSIS via Traffic Analyzer
# ---------------------------------------------------------

unique_tx_set, node_counts, files_count = traffic_analyzer.count_validated_transactions(LOG_DIR, ALL_NODES)

# Print per-node statistics
for node in ALL_NODES:
    count = node_counts.get(node, 0)
    print(f"{node}: {count} distinct Validated transactions")

TOTAL_UNIQUE_TXS = len(unique_tx_set)
print("\n" + "=" * 50)
print(f"TOTAL DISTINCT TRANSACTIONS ACROSS ALL NODES: {TOTAL_UNIQUE_TXS}")
print("=" * 50)



## 2. Traffic Analysis

Analyzes block production to calculate:
- Total validated transactions
- Transactions per second (TPS)
- Block finalization times

In [ ]:
#===================================================================================
# Count Extrinsics, Max Txs, & Calculate Finalization Duration
#===================================================================================

# ---------------------------------------------------------
# STEP 2: ANALYSIS LOGIC via Traffic Analyzer
# ---------------------------------------------------------

tx_counts, creation_times, finalization_times = traffic_analyzer.analyze_block_production(LOG_DIR, BLOCK_PRODUCERS)

# ---------------------------------------------------------
# STEP 3: CALCULATE & PRINT
# ---------------------------------------------------------

traffic_stats = traffic_analyzer.print_traffic_report(tx_counts, creation_times, finalization_times)
TOTAL_TXS_VALIDATED = traffic_stats["total_txs"]
MAX_TXS_BLOCK = traffic_stats["max_txs_block"]
TOTAL_TX_PROCESSING_TIME = traffic_stats["duration"]
START_TIME = traffic_stats["start_time"]
END_TIME = traffic_stats["end_time"]


## 3. Mempool Analysis

Examines transaction pool behavior:
- Ready vs future transactions
- Admission rates
- Validation/revalidation/pruning lifecycle
- Throughput and mempool depth visualizations

In [ ]:
# ==========================================
# MEMPOOL METRICS ANALYSIS
# ==========================================
print(f"Scanning TxPool metrics for specific nodes: {TX_POOL_NODES}...")

# Using mempool_benchmarks extractor to parse logs for all nodes at once
all_mempool_events = extractor.parse_logs(TX_POOL_NODES, log_dir=LOG_DIR)

from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------
# 1. INDIVIDUAL NODE ANALYSIS
# ------------------------------------------
for node_name in TX_POOL_NODES:
    print(f"\n" + "=" * 50)
    print(f" ANALYSIS FOR NODE: {node_name}")
    print("=" * 50)

    # Filter events for this node
    node_events = [e for e in all_mempool_events if e.node == node_name]
    
    if not node_events:
        print(f"No mempool events found for node {node_name}.")
        continue

    # Convert to Points for analyzer
    mempool_points = [
        analyzer.MempoolPoint(
            timestamp=e.timestamp,
            node=e.node,
            ready=e.ready,
            future=e.future,
            mempool_len=e.mempool_len,
            submitted_count=e.submitted_count,
            validated_count=e.validated_count,
            revalidated=e.revalidated,
            pruned_count=e.pruned_count,
            reverified_txs=e.reverified_txs
        ) for e in node_events
    ]

    # Analyze
    window_ms = 1000
    resampled_df = analyzer.resample_metrics(analyzer.to_dataframe(mempool_points), window_ms)
    original_df = analyzer.to_dataframe(mempool_points)

    print("\n MEMPOOL ANALYSIS SUMMARY")
    print("-" * 30)
    print(analyzer.summarize(resampled_df, original_df))
    print("-" * 30)
    print(analyzer.generate_insights(resampled_df, original_df))
    
    # Calculate max mempool depth
    max_depth = 0
    for p in mempool_points:
        if p.mempool_len and p.mempool_len > max_depth:
             max_depth = p.mempool_len
    print(f"Max Mempool Depth for {node_name}: {max_depth}")
    
    # --- CHART 1: Throughput & Depth ---
    plot_filename = f'mempool_analysis_{node_name}.png'
    print(f"\n GENERATING THROUGHPUT/DEPTH VISUALIZATION: {plot_filename}")
    try:
        analyzer.plot_throughput_and_mempool(resampled_df, original_df, plot_filename)
        print(f"Graph saved: {plot_filename}")
        display(Image(filename=plot_filename))
    except Exception as e:
        print(f"Warning: Could not generate mempool plots for {node_name}: {e}")

    # --- CHART 2: Lifecycle Events (Revalidation/Pruning) ---
    lifecycle_filename = f'lifecycle_analysis_{node_name}.png'
    print(f" GENERATING LIFECYCLE VISUALIZATION: {lifecycle_filename}")
    try:
        lf_df = original_df.copy()
        if 'timestamp' in lf_df.columns:
            lf_df.set_index('timestamp', inplace=True)
        
        # Columns to plot
        cols = ['revalidated', 'pruned', 'validated']
        cols = [c for c in cols if c in lf_df.columns]
        
        if cols:
            # Resample sum over window
            lf_res = lf_df[cols].resample(f'{window_ms}ms').sum()
            
            # Convert to TPS
            factor = 1000.0 / window_ms
            lf_rate = lf_res * factor
            
            fig_lf, ax_lf = plt.subplots(figsize=(14, 6))
            
            if 'revalidated' in lf_rate:
                ax_lf.plot(lf_rate.index, lf_rate['revalidated'], label='Revalidated (Tx/s)', color='orange', alpha=0.8)
            if 'pruned' in lf_rate:
                ax_lf.plot(lf_rate.index, lf_rate['pruned'], label='Pruned (Finalized) (Tx/s)', color='green', alpha=0.8)
            if 'validated' in lf_rate:
                ax_lf.plot(lf_rate.index, lf_rate['validated'], label='New Validated (Tx/s)', color='blue', alpha=0.6, linestyle=':')

            ax_lf.set_title(f'Transaction Lifecycle Rates ({node_name})')
            ax_lf.set_ylabel('Rate (Tx/s)')
            ax_lf.set_xlabel('Time (UTC)')
            ax_lf.legend(loc='upper left')
            ax_lf.grid(True, alpha=0.3)
            ax_lf.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            
            plt.tight_layout()
            plt.savefig(lifecycle_filename, dpi=100)
            plt.close(fig_lf)
            
            print(f"Graph saved: {lifecycle_filename}")
            display(Image(filename=lifecycle_filename))
    except Exception as e:
        print(f"Could not generate lifecycle plot: {e}")

# ------------------------------------------
# 2. COMBINED ANALYSIS (All Nodes)
# ------------------------------------------
print("\n" + "=" * 50)
print(" COMBINED MEMPOOL ANALYSIS")
print("=" * 50)

if all_mempool_events:
    # Convert ALL events to points
    all_points = [
        analyzer.MempoolPoint(
            timestamp=e.timestamp,
            node=e.node,
            ready=e.ready,
            future=e.future,
            mempool_len=e.mempool_len,
            submitted_count=e.submitted_count,
            validated_count=e.validated_count,
            revalidated=e.revalidated,
            pruned_count=e.pruned_count,
            reverified_txs=e.reverified_txs
        ) for e in all_mempool_events
    ]

    # Analyze combined
    window_ms = 1000
    resampled_all = analyzer.resample_metrics(analyzer.to_dataframe(all_points), window_ms)
    original_all = analyzer.to_dataframe(all_points)
    
    # Plot combined
    combined_plot_filename = 'mempool_analysis_combined.png'
    print(f"\n GENERATING COMBINED VISUALIZATION: {combined_plot_filename}")
    try:
        analyzer.plot_throughput_and_mempool(resampled_all, original_all, combined_plot_filename)
        print(f"Graph saved: {combined_plot_filename}")
        display(Image(filename=combined_plot_filename))
    except Exception as e:
        print(f"Warning: Could not generate combined plot: {e}")
else:
    print("No events available for combined analysis.")


## Benchmark Summary

Visual summary of mempool depth and transaction throughput over time.

In [ ]:
# ==========================================
# BENCHMARK SUMMARY PLOT
# ==========================================
# This creates a 2-subplot figure with:
# 1. Mempool Depth Over Time
# 2. Cumulative Throughput

# 1. Prepare Mempool Data for Plot
# Convert mempool_points to a DataFrame format suitable for plotting
df_pool_plot = []
if mempool_points:
    for p in mempool_points:
        df_pool_plot.append({
            'time': p.timestamp,
            'node': p.node,
            'pool_count': p.mempool_len if p.mempool_len else 0
        })

df_pool_window = pd.DataFrame(df_pool_plot)

# 2. Prepare Block Data for Throughput Plot
block_data = []
for blk, count in tx_counts.items():
    if blk in creation_times:
        block_data.append({'time': creation_times[blk], 'tx_count': count})

df_blocks = pd.DataFrame(block_data).sort_values(by='time')
if not df_blocks.empty:
    df_blocks['cumulative_txs'] = df_blocks['tx_count'].cumsum()

# Create Figure with 2 Subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=False)

# PLOT 1: Mempool Depth Over Time
if not df_pool_window.empty:
    sns.lineplot(data=df_pool_window, x='time', y='pool_count', hue='node', marker='o', ax=ax1, errorbar=None)
    ax1.set_title('Mempool Depth (Pending Transactions) Over Time', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Pending Txs')
    ax1.set_xlabel('Time')
    ax1.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax1.legend(title='Node')
    
else:
    ax1.text(0.5, 0.5, 'No Mempool Data Found', horizontalalignment='center', verticalalignment='center')
    ax1.set_title('Mempool Depth (No Data)', fontsize=14, fontweight='bold')

# PLOT 2: Cumulative Throughput
if not df_blocks.empty:
    ax2.plot(df_blocks['time'], df_blocks['cumulative_txs'], marker='s', color='green', linestyle='-', label='Processed Txs')
    ax2.set_title('Throughput: Cumulative Processed Transactions', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Total Finalized Transactions')
    ax2.set_xlabel('Time')
    ax2.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax2.legend()
    
    # Annotate final count
    last_time = df_blocks['time'].iloc[-1]
    last_count = df_blocks['cumulative_txs'].iloc[-1]
    ax2.annotate(f"{last_count} Txs", (last_time, last_count), textcoords="offset points", xytext=(-10,10), ha='center')
else:
    ax2.text(0.5, 0.5, 'No Block Transaction Data', horizontalalignment='center', verticalalignment='center')

plt.tight_layout()
plt.savefig('benchmark_summary_plot.png')
print("Plot saved as 'benchmark_summary_plot.png'")
plt.show()


## 4. Validation Timing

Detailed analysis of the transaction validation pipeline:
- Per-stage timing breakdown
- Cumulative processing times
- Timeline visualization

In [ ]:
import numpy as np
# MIDNIGHT LEDGER v2 LOGS ANALYSIS
# ==============================================================================
# CONFIGURATION
# ==============================================================================
# List of log files to analyze (add your file paths here)
LOG_FILES = []
for node in BLOCK_PRODUCERS:
    file_path = os.path.join(LOG_DIR, f"{node}.txt")
        
    if not os.path.exists(file_path):       
        # print(f"⚠️ Log file not found for: {node}") # Optional warning
        continue
    LOG_FILES.append(file_path)

# ==============================================================================
# 2. PARSING LOGIC
# ==============================================================================
def parse_cumulative_timings(files):
    """
    Parses logs for lines like:
    INFO ... midnight::ledger_v2: ⏱️  Step Name ... (elapsed_ms=X)
    """
    # Regex to capture:
    # Group 1: The Step Name (e.g. "Deserializing tx")
    # Group 2: The Cumulative Time (elapsed_ms)
    rx_timing = re.compile(r'midnight::ledger_v2: ⏱️\s+(.*?)\s+(?:.*elapsed_ms=(\d+))')
    
    category_data = {}

    print(f"Scanning {len(files)} files for ledger timings...")

    for file_path in files:
        if not os.path.exists(file_path):
            print(f"Skipping missing file: {file_path}")
            continue
            
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                match = rx_timing.search(line)
                if match:
                    step_name = match.group(1).strip()
                    # Clean up any trailing punctuation if strictly text
                    if step_name.endswith(','): step_name = step_name[:-1]
                    
                    try:
                        duration = int(match.group(2))
                    except ValueError:
                        continue
                    
                    if step_name not in category_data:
                        category_data[step_name] = []
                    category_data[step_name].append(duration)

    return category_data

# ==============================================================================
# 3. STATS CALCULATION
# ==============================================================================
def calculate_statistics(data_dict):
    if not data_dict:
        return pd.DataFrame()

    stats_list = []
    
    # Calculate stats for each individual step
    for step, values in data_dict.items():
        if not values:
            continue
        
        arr = np.array(values)
        stats_list.append({
            'Step': step,
            'Average': np.mean(arr),
            'p05': np.percentile(arr, 5),
            'p95': np.percentile(arr, 95),
            'Count': len(arr)
        })

    df = pd.DataFrame(stats_list)
    
    # Sort by Average time to reconstruct the chronological flow
    if not df.empty:
        df = df.sort_values(by='Average', ascending=True)

    return df

# ==============================================================================
# 4. PLOTTING
# ==============================================================================
def plot_validation_timeline(df):
    if df.empty:
        print("No data found to plot.")
        return

    steps = df['Step']
    averages = df['Average']
    
    # Identify the final step (Total Time) for highlighting
    # Usually 'UTXO integrity ok' based on your logs, or just the one with max time
    final_step_idx = df['Average'].idxmax() 
    
    # Error bars (relative to the average)
    yerr_lower = (averages - df['p05']).clip(lower=0)
    yerr_upper = (df['p95'] - averages).clip(lower=0)
    error_bars = [yerr_lower, yerr_upper]

    plt.figure(figsize=(14, 7))
    
    # Create bars
    # Using a list of colors to highlight the "Total" (last step)
    colors = ['skyblue'] * len(df)
    # Highlight the specific "UTXO integrity ok" step if present, or just the last one
    for i, step in enumerate(steps):
        if "integrity ok" in step or i == len(steps) - 1:
            colors[i] = 'orange'

    bars = plt.bar(steps, averages, yerr=error_bars, capsize=5, 
                   color=colors, edgecolor='black', alpha=0.8)

    plt.title('Tx Validation Timeline: Cumulative Time to Reach Each Step')
    plt.ylabel('Cumulative Time (ms)')
    plt.xlabel('Validation Step')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    # Add numeric labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        # Label Avg
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.2, 
                 f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    output_file = 'tx_validation_timeline.png'
    plt.savefig(output_file)
    print(f"\nPlot saved to {output_file}")
    # plt.show() # Uncomment if running locally

# ==============================================================================
# EXECUTION
# ==============================================================================
raw_data = parse_cumulative_timings(LOG_FILES)
stats_df = calculate_statistics(raw_data)

if not stats_df.empty:
    print("\n--- Validation Pipeline Stats (Cumulative ms) ---")
    # Display table
    print(stats_df[['Step', 'p05', 'Average', 'p95', 'Count']].to_string(index=False))
    
    # Print the specific "Total" the user requested
    total_row = stats_df[stats_df['Step'].str.contains("integrity ok", case=False)]
    if not total_row.empty:
        avg_total = total_row['Average'].values[0]
        print(f"\n✅ TOTAL VALIDATION TIME (Avg): {avg_total:.2f} ms")
    
    plot_validation_timeline(stats_df)
else:
    print("No matching ledger logs found.")

In [ ]:
#===================================================================================\
# Plot block production timeline
#===================================================================================\\


# ---------------------------------------------------------
# STEP 4: PARSE BLOCK CREATION TIMES (Re-parsing for Timeline)
# ---------------------------------------------------------
# We need to scan the logs again specifically for block creation events 
# to build the creation_registry required for the plot.

creation_registry = {} 

# Regex to capture timestamp and creation event
timestamp_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)")
creation_pattern = re.compile(r"Pre-sealed block for proposal at (\d+)")

print(f"Scanning for block creation events in: {LOG_DIR}")

for node in BLOCK_PRODUCERS:
    filepath = os.path.join(LOG_DIR, f"{node}.txt")
    
    if not os.path.exists(filepath):
        continue
        
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            
            # Check for creation line first (optimization)
            create_match = creation_pattern.search(line)
            if create_match:
                # Extract Timestamp
                ts_match = timestamp_pattern.search(line)
                if not ts_match: continue
                
                try:
                    ts_str = ts_match.group(1)
                    if len(ts_str.split('.')[-1]) > 6: ts_str = ts_str[:26]
                    current_time = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")
                    
                    block_num = int(create_match.group(1))
                    
                    # Store data
                    creation_registry[block_num] = {
                        'time': current_time,
                        'creator': node
                    }
                except ValueError:
                    continue

# ---------------------------------------------------------
# STEP 5: PLOT BLOCK CREATION TIMELINE
# ---------------------------------------------------------

if not creation_registry:
    print("No block creation events found. Check log path.")
else:
    # Convert registry to DataFrame
    creation_data = []
    for block_num, info in creation_registry.items():
        creation_data.append({
            "Block": block_num,
            "Timestamp": info['time'],
            "Creator": info['creator']
        })

    df_creation = pd.DataFrame(creation_data).sort_values("Block")

    print(f"Found {len(df_creation)} blocks. Plotting...")

    # Setup the Plot
    plt.figure(figsize=(14, 7))
    sns.set_theme(style="whitegrid")

    # 1. Draw a line to show the progression
    sns.lineplot(
        data=df_creation,
        x="Block",
        y="Timestamp",
        color="gray",
        alpha=0.3,
        zorder=1
    )

    # 2. Draw points colored by the Creator Node
    plot = sns.scatterplot(
        data=df_creation,
        x="Block",
        y="Timestamp",
        hue="Creator",
        palette="tab20",  # Colorful palette for many nodes
        s=100,            # Dot size
        edgecolor="black",
        zorder=2
    )

    # Format Y-Axis to show HH:MM:SS
    plot.yaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    plt.title("Block Creation Timeline", fontsize=16)
    plt.xlabel("Block Number", fontsize=12)
    plt.ylabel("Creation Time (UTC)", fontsize=12)

    # Ensure integer ticks on X-axis if valid range
    if df_creation["Block"].nunique() < 20:
        plt.xticks(df_creation["Block"].unique())

    # Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title="Producer")
    
    plt.tight_layout()
    plt.savefig("block_creation_timeline.png", dpi=300)
    print("Graph saved as 'block_creation_timeline.png'")
    
    plt.show()

In [ ]:
#===================================================================================
# Plot all Block Producer's block production time and the start delay
#===================================================================================

files_to_process = []
for node in BLOCK_PRODUCERS:
    # os.path.join automatically adds the '/' if LOG_DIR doesn't have it
    files_to_process.append(os.path.join(LOG_DIR, f"{node}.txt"))

# # ---------------------------------------------------------
# # STEP 6: PARSE THE LOGS
# # ---------------------------------------------------------

def parse_substrate_logs_robust(file_list):
    data = []
    
    # 1. Regex to capture Timestamp and the rest of the message
    # Matches: 2026-01-15 17:39:36.048  INFO ...
    base_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\s+(.*)")
    
    # 2. Regex for START event (captures Parent Block #)
    # Matches: ... Starting consensus session on top of parent ... (#14406)
    start_pattern = re.compile(r"Starting consensus session on top of parent .* \(#(\d+)\)")
    
    # 3. Regex for END event (captures Current Block #)
    # Matches: ... Prepared block for proposing at 14407 ...
    end_pattern = re.compile(r"Prepared block for proposing at (\d+)")

    for filename in file_list:
        node_name = os.path.splitext(os.path.basename(filename))[0].capitalize()
        
        # Dictionary to store timestamps for each block
        # Structure: { 14407: {'start': datetime, 'end': datetime}, ... }
        block_events = {}
        
        with open(filename, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                
                # Extract Timestamp
                base_match = base_pattern.search(line)
                if not base_match: continue
                
                timestamp_str = base_match.group(1)
                message = base_match.group(2)
                
                try:
                    current_time = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S.%f")
                except ValueError:
                    continue

                # --- CHECK FOR START (Parent Block) ---
                start_match = start_pattern.search(message)
                if start_match:
                    parent_block = int(start_match.group(1))
                    # If we are working on top of 14406, we are building 14407
                    target_block = parent_block + 1
                    
                    if target_block not in block_events: block_events[target_block] = {}
                    block_events[target_block]['start'] = current_time
                    continue

                # --- CHECK FOR END (Current Block) ---
                end_match = end_pattern.search(message)
                if end_match:
                    target_block = int(end_match.group(1))
                    
                    if target_block not in block_events: block_events[target_block] = {}
                    block_events[target_block]['end'] = current_time
                    continue

        # --- CALCULATE DURATIONS ---
        # Now we iterate through the dictionary and calculate stats for blocks that have both times
        for block_num, times in block_events.items():
            if 'start' in times and 'end' in times:
                start_time = times['start']
                end_time = times['end']
                
                # A. Production Time
                # Calculate diff in milliseconds
                duration_ms = (end_time - start_time).total_seconds() * 1000
                
                # Sanity check: If duration is negative (rare clock skew?), skip
                if duration_ms < 0: continue

                # B. Start Delay (Ideal Target vs Actual Start)
                # Calculate seconds past the minute
                total_seconds = start_time.second + (start_time.microsecond / 1e6)
                remainder = total_seconds % 6 
                start_delay_ms = remainder * 1000
                
                data.append({
                    "Node": node_name,
                    "Block": block_num,
                    "ProductionTime_ms": duration_ms,
                    "StartDelay_ms": start_delay_ms
                })

    return pd.DataFrame(data)

# Run the parser
df = parse_substrate_logs_robust(files_to_process)

print(f"Parsed {len(df)} blocks successfully.")
if not df.empty:
    print(df.head())
else:
    print("No matching blocks found. Check log paths or regex.")

# ---------------------------------------------------------
# STEP 3: PLOT THE GRAPHS
# ---------------------------------------------------------

if not df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    sns.set_theme(style="whitegrid")

    # GRAPH 1: Production Time (Duration)
    # How long did the CPU actually churn to build the block?
    sns.lineplot(
        ax=axes[0],
        data=df, 
        x="Block", 
        y="ProductionTime_ms", 
        hue="Node", 
        marker="o",
        palette="tab10"
    )
    axes[0].set_title("1. Actual Block Production Duration (End - Start)", fontsize=14, fontweight='bold')
    axes[0].set_ylabel("Duration (ms)")
    axes[0].legend(loc='upper right')

    # GRAPH 2: Start Delay
    # How long after the ideal 6s slot did the node start working?
    sns.lineplot(
        ax=axes[1],
        data=df, 
        x="Block", 
        y="StartDelay_ms", 
        hue="Node", 
        marker="s", # Square marker to distinguish
        palette="tab10"
    )
    axes[1].set_title("2. Start Delay (Actual Start - Target Slot Time)", fontsize=14, fontweight='bold')
    axes[1].set_ylabel("Delay (ms)")
    axes[1].set_xlabel("Block Number")
    
    # Force Integer X-Axis
    if len(df) < 20:
        axes[1].set_xticks(sorted(df['Block'].unique()))

    plt.tight_layout()
    plt.savefig("block_timing_analysis.png")
    plt.show()
else:
    print("No matching log lines found. Check regex patterns.")

In [ ]:
#===================================================================================
# Plot each Block Producer's block production time
#===================================================================================

# 1. Get the list of nodes
unique_nodes = sorted(df['Node'].unique())

# 2. Determine global X-axis limits
min_block = df['Block'].min()
max_block = df['Block'].max()

# 3. Determine global Y-axis limit (New Step)
# Find the max time across ALL nodes and add a 10% buffer for visual spacing
max_y_val = df['ProductionTime_ms'].max()
global_y_limit = max_y_val * 1.1 

# 4. Iterate and plot
for node in unique_nodes:
    # Filter data for this specific node
    node_data = df[df['Node'] == node]
    
    # Skip if no data for this node
    if node_data.empty:
        continue

    # Create a new figure for each node
    plt.figure(figsize=(10, 3))
    
    # Plot Production Time
    sns.lineplot(
        data=node_data, 
        x="Block", 
        y="ProductionTime_ms", 
        marker="o", 
        color="#2ca02c"
    )
    
    # Formatting
    plt.title(f"Production Time (End - Start) for: {node}", fontsize=14, fontweight='bold')
    plt.ylabel("Time (ms)")
    plt.xlabel("Block Number")
    
    # --- GLOBAL AXIS LIMITS ---
    # 1. Apply global X limits
    plt.xlim(min_block, max_block)
    
    # 2. Apply global Y limits (0 to Global Max)
    plt.ylim(0, global_y_limit)
    # --------------------------
    
    # Handle ticks: If the total range is small, show every integer. 
    if (max_block - min_block) < 50:
        plt.xticks(range(int(min_block), int(max_block) + 1))
        
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------
# Plot all block import times
# ---------------------------------------------------------




# ---------------------------------------------------------
# STEP 2: HELPER FUNCTIONS
# ---------------------------------------------------------

def to_short_hash(full_hash):
    """
    Converts a full hash (e.g. 0xf2d38b...ed171) to the log format (0xf2d3…d171).
    Standard Substrate log format is usually: 0x + first 4 + … + last 4
    """
    if not full_hash.startswith("0x"):
        return full_hash
    
    # Remove 0x prefix for slicing
    clean = full_hash[2:]
    if len(clean) < 8: return full_hash
    
    return f"0x{clean[:4]}…{clean[-4:]}"

# ---------------------------------------------------------
# STEP 3: PARSE THE FILES
# ---------------------------------------------------------

# Dictionary to store creation times keyed by the BLOCK HASH (Short format)
# Key: ShortHash (str), Value: { 'time': datetime, 'block_num': int, 'node': str }
creation_registry = {}

# List to store raw import events to be processed later
# Item: { 'time': datetime, 'block_num': int, 'short_hash': str, 'node': str }
import_events = []

# Regex patterns
timestamp_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)")

# Capture Creation: "Pre-sealed block for proposal at 3040. Hash now 0xf2d3..."
# We use the "Pre-sealed" line because it contains the final finalized hash.
creation_pattern = re.compile(r"Pre-sealed block for proposal at (\d+)\. Hash now (0x[a-fA-F0-9]+)")

# Capture Import: "Imported #3040 (0x4e7c…cc64 → 0xf2d3…d171)"
# We capture Group 1 (Block Num) and Group 2 (The second hash, which is the new block hash)
import_pattern = re.compile(r"Imported #(\d+) \(.*? → (0x[a-fA-F0-9…\.]+)\)")

print(f"Reading logs from: {LOG_DIR}")

for node in ALL_NODES:
    filepath = os.path.join(LOG_DIR, f"{node}.txt")
    
    if not os.path.exists(filepath):
        print(f"  - Warning: Log file not found for {node}")
        continue
        
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            # 1. Extract Timestamp
            ts_match = timestamp_pattern.search(line)
            if not ts_match: continue
            
            try:
                ts_str = ts_match.group(1)
                # Truncate microseconds if too long (Python limit)
                if len(ts_str.split('.')[-1]) > 6: ts_str = ts_str[:26] 
                current_time = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")
            except ValueError:
                continue 

            # 2. Check for BLOCK CREATION (The Source of Truth)
            create_match = creation_pattern.search(line)
            if create_match:
                block_num = int(create_match.group(1))
                full_hash = create_match.group(2)
                short_hash = to_short_hash(full_hash)
                
                # We store this hash as the canonical creation event
                creation_registry[short_hash] = {
                    'time': current_time,
                    'block_num': block_num,
                    'creator': node
                }
                continue

            # 3. Check for BLOCK IMPORT
            import_match = import_pattern.search(line)
            if import_match:
                block_num = int(import_match.group(1))
                imported_hash = import_match.group(2)
                
                import_events.append({
                    'time': current_time,
                    'block_num': block_num,
                    'short_hash': imported_hash,
                    'node': node
                })

print(f"Processing complete.")
print(f"  - Unique Blocks Created: {len(creation_registry)}")
print(f"  - Total Import Events: {len(import_events)}")

# ---------------------------------------------------------
# STEP 4: CALCULATE LATENCY (MATCHING BY HASH)
# ---------------------------------------------------------

data = []

for event in import_events:
    target_hash = event['short_hash']
    importer = event['node']
    import_time = event['time']
    
    # Only calculate if we have a creation record for THIS SPECIFIC HASH
    if target_hash in creation_registry:
        genesis_info = creation_registry[target_hash]
        creation_time = genesis_info['time']
        creator = genesis_info['creator']
        block_num = genesis_info['block_num']
        
        # If this node is the creator, latency is 0
        if importer == creator:
            delay_ms = 0.0
        else:
            delta = import_time - creation_time
            delay_ms = delta.total_seconds() * 1000.0
        
        data.append({
            "Block": block_num,
            "Node": importer,
            "ImportTime_ms": delay_ms,
            "Hash": target_hash
        })

# ---------------------------------------------------------
# STEP 5: PLOT
# ---------------------------------------------------------

if not data:
    print("No matching hash data found! Check if the log format matches the patterns.")
else:
    df = pd.DataFrame(data)
    df = df.sort_values(by="Block")
    
    # Display sample to verify
    print("\nSample Data (First 5 rows):")
    print(df.head())

    plt.figure(figsize=(14, 7))
    sns.set_theme(style="whitegrid")

    sns.lineplot(
        data=df, 
        x="Block", 
        y="ImportTime_ms", 
        hue="Node", 
        marker="o",
        palette="tab20", 
        linewidth=1.5,
        errorbar=None 
    )

    plt.title("Block Propagation Delay", fontsize=16)
    plt.xlabel("Block Number", fontsize=12)
    plt.ylabel("Propagation Delay (ms)", fontsize=12)
    
    # Formatting X-axis
    if df["Block"].nunique() < 20:
        plt.xticks(df["Block"].unique())

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title="Node")
    plt.tight_layout()
    
    plt.savefig("import_times_hash_matched.png", dpi=300)
    print("\nGraph saved as 'import_times_hash_matched.png'")
    
    plt.show()

In [ ]:
# ---------------------------------------------------------
# Plot block import times per node
# ---------------------------------------------------------


# 1. PRE-CALCULATE THE Y-AXIS LIMIT
# Find the max value in the whole dataset and add a 10% buffer for visual spacing
max_val = df['ImportTime_ms'].max()
global_y_limit = max_val * 1.1 

# Get a list of all unique nodes and sort them alphabetically
nodes = sorted(df['Node'].unique())

# Loop through each node and create a separate graph
for node in nodes:
    # Filter the data for just this node
    node_data = df[df['Node'] == node]
    
    # Create a new figure for this node
    plt.figure(figsize=(12, 3)) 
    
    # Plot the line
    sns.lineplot(
        data=node_data, 
        x="Block", 
        y="ImportTime_ms", 
        marker="o",
        color='tab:blue',
        errorbar=None
    )
    
    # Formatting
    plt.title(f"Import Times for: {node.upper()}", fontsize=14, fontweight='bold')
    plt.ylabel("Time (ms)")
    plt.xlabel("Block Number")
    
    # --- CRITICAL CHANGE HERE ---
    # Force every plot to use the same 0 to Global Max scale
    plt.ylim(0, global_y_limit)
    # ----------------------------
    
    # Ensure X-axis shows every block integer (if list isn't too long)
    if len(df["Block"].unique()) < 50:
        plt.xticks(df["Block"].unique())
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    # Display the graph
    plt.show()

## 5. Block Size Analysis

Fetches block data from the node and analyzes:
- Block size distribution
- Size vs extrinsic count correlation
- Largest blocks identification

In [ ]:

# FORCE RELOAD to pick up any changes to the scripts on disk
importlib.reload(fetch_block_sizes)
importlib.reload(extract_block_range_from_logs)

# ==========================================
# CONFIGURATION
# ==========================================
NODE_URL = "ws://george.node.sc.iog.io:9944"

print(f"Using LOG_DIR: {LOG_DIR}")

# AUTOMATIC RANGE DETECTION
# If valid block range is found from logs, we prioritize that over LATEST_N
try:
    print("Extracting block range from logs...")
    
    from datetime import datetime, timezone, timedelta
    
    # Default to a wide range in UTC
    start_time = datetime(2020, 1, 1, tzinfo=timezone.utc)
    end_time = datetime(2030, 1, 1, tzinfo=timezone.utc)
    
    # Try to parse from directory name 'from_YYYY..._to_YYYY...'
    log_dir_name = os.path.basename(LOG_DIR.rstrip('/'))
    if log_dir_name.startswith('from_') and '_to_' in log_dir_name:
        try:
            parts = log_dir_name.split('_to_')
            start_str = parts[0].replace('from_', '')
            end_str = parts[1]
            
            # Assume directory times are EST (UTC-5) as per common workflow
            est_tz = timezone(timedelta(hours=-5))
            
            # Format: YYYY-MM-DD_HH-MM-SS
            dt_start = datetime.strptime(start_str, "%Y-%m-%d_%H-%M-%S").replace(tzinfo=est_tz)
            dt_end = datetime.strptime(end_str, "%Y-%m-%d_%H-%M-%S").replace(tzinfo=est_tz)
            
            # Convert to UTC
            start_time = dt_start.astimezone(timezone.utc)
            end_time = dt_end.astimezone(timezone.utc)
            
            print(f"Inferred time range from directory (EST -> UTC):")
            print(f"  Input: {start_str} to {end_str} (EST)")
            print(f"  UTC:   {start_time} to {end_time}")
        except ValueError as e:
            print(f"Could not parse time range from directory name ({e}), using default wide range.")

    # Find the range
    start_block, end_block = extract_block_range_from_logs.find_block_range_from_logs(
        LOG_DIR, start_time, end_time, min_block_threshold=0
    )
    
    print(f"Found block range from logs: #{start_block} to #{end_block}")
    
    # Disable LATEST_N if range is found
    LATEST_N = None
    START_BLOCK = start_block
    END_BLOCK = end_block
    
except Exception as e:
    print(f"Error extracting block range from logs: {e}")
    print("Falling back to LATEST_N configuration.")
    LATEST_N = 100
    START_BLOCK = None
    END_BLOCK = None

# ==========================================
# FETCH DATA
# ==========================================
try:
    print(f"Connecting to node at {NODE_URL}...")
    substrate = fetch_block_sizes.connect_to_node(NODE_URL)
    
    if LATEST_N:
        print(f"Fetching latest {LATEST_N} blocks...")
        block_data = fetch_block_sizes.fetch_latest_n_blocks(substrate, LATEST_N)
    elif START_BLOCK is not None and END_BLOCK is not None:
        print(f"Fetching blocks #{START_BLOCK} to #{END_BLOCK} ({END_BLOCK - START_BLOCK + 1} blocks)...")
        block_data = fetch_block_sizes.fetch_block_range(substrate, START_BLOCK, END_BLOCK)
    else:
        print("No valid block range or LATEST_N specified.")
        block_data = []

    substrate.close()
    
    # Convert to DataFrame
    df = pd.DataFrame(block_data)
    
    # Display info
    print(f"\nSuccessfully loaded {len(df)} blocks")
    if not df.empty:
        print(f"Block range: #{df['block_number'].min()} to #{df['block_number'].max()}")
        display(df.head())
    else:
        print("No block data fetched.")

except Exception as e:
    print(f"Error fetching data: {e}")
    import traceback
    traceback.print_exc()
    # Create empty dataframe to prevent errors in subsequent cells
    df = pd.DataFrame(columns=['block_number', 'block_hash', 'size_bytes', 'size_kb', 'size_mb', 'extrinsic_count', 'timestamp'])

## Summary Statistics

In [ ]:
print("=" * 60)
print("BLOCK SIZE STATISTICS")
print("=" * 60)
print(f"Average block size:     {df['size_kb'].mean():.2f} KB")
print(f"Median block size:      {df['size_kb'].median():.2f} KB")
print(f"Min block size:         {df['size_kb'].min():.2f} KB")
print(f"Max block size:         {df['size_kb'].max():.2f} KB")
print(f"Std deviation:          {df['size_kb'].std():.2f} KB")
print(f"Total data:             {df['size_mb'].sum():.2f} MB")
print()
print("EXTRINSIC STATISTICS")
print("=" * 60)
print(f"Total extrinsics:       {df['extrinsic_count'].sum():,}")
print(f"Average per block:      {df['extrinsic_count'].mean():.1f}")
print(f"Min per block:          {df['extrinsic_count'].min()}")
print(f"Max per block:          {df['extrinsic_count'].max()}")

## Block Size Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df['block_number'], df['size_kb'], marker='o', markersize=4, linewidth=1.5, alpha=0.7)
ax.axhline(y=df['size_kb'].mean(), color='r', linestyle='--', linewidth=2, label=f"Average: {df['size_kb'].mean():.2f} KB")

ax.set_xlabel('Block Number', fontsize=12)
ax.set_ylabel('Block Size (KB)', fontsize=12)
ax.set_title('Block Size Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Block Size Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
ax1.hist(df['size_kb'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(df['size_kb'].mean(), color='r', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(df['size_kb'].median(), color='g', linestyle='--', linewidth=2, label='Median')
ax1.set_xlabel('Block Size (KB)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Block Size Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Box plot
ax2.boxplot(df['size_kb'], vert=True, patch_artist=True, 
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
ax2.set_ylabel('Block Size (KB)', fontsize=12)
ax2.set_title('Block Size Box Plot', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Top 10 Largest Blocks

In [ ]:
top_blocks = df.nlargest(10, 'size_kb')[['block_number', 'size_kb', 'extrinsic_count']]
top_blocks.reset_index(drop=True, inplace=True)
top_blocks.index = top_blocks.index + 1
print("Top 10 Largest Blocks:")
print(top_blocks.to_string())

## Size per Extrinsic Analysis

In [ ]:
# Calculate average size per extrinsic for each block
df['size_per_extrinsic'] = df['size_kb'] / df['extrinsic_count'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df['block_number'], df['size_per_extrinsic'], marker='o', markersize=4, linewidth=1.5, alpha=0.7, color='orange')
ax.axhline(y=df['size_per_extrinsic'].mean(), color='r', linestyle='--', linewidth=2, 
           label=f"Average: {df['size_per_extrinsic'].mean():.2f} KB/extrinsic")

ax.set_xlabel('Block Number', fontsize=12)
ax.set_ylabel('KB per Extrinsic', fontsize=12)
ax.set_title('Average Size per Extrinsic Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Fork Analysis

Detects and analyzes chain reorganizations and competing blocks.

In [ ]:
# Fork Analysis using extracted module
print('Analyzing Forks...')
if 'ALL_NODES' not in locals():
    ALL_NODES = BLOCK_PRODUCERS + RELAYS
fork_analyzer.analyze_forks(LOG_DIR, ALL_NODES)
